# Next Commander Experiment: More Data And Paired Evaluation
Run this notebook in VS Code using the official Colab CPU kernel. No local simulation/training. Upload `runs/cedhai-next.zip` through Explorer > Upload to Colab once. The reviewed checkpoint is recovered from your existing Drive folder, so you do not need to upload it again.

Stages: 24 new collection games (six seeds), dataset audit + 20 training epochs, 20 reference games and 20 candidate games on matched fresh seeds, then comparison export. This may require multiple Colab sessions. Drive retains stage state and partial attempts. Training has five independent training seeds but only one validation seed: useful for expansion, still a limited generalization check.

Run cells individually in order. Do not use Run All before checking the dataset warnings. If interrupted, reconnect to Colab, use the same bundle and RUN_NAME, and rerun setup plus the interrupted stage. Do not edit package code or parameters mid-experiment.


In [ ]:
from pathlib import Path
import platform
if platform.system() != 'Linux' or not Path('/content').is_dir():
    raise RuntimeError('Select the remote Colab kernel, not local Python')
from google.colab import drive
drive.mount('/content/drive')
RUN_NAME = 'expanded-v1'
run_root = Path('/content/drive/MyDrive/cedhai/cloud') / RUN_NAME
run_root.mkdir(parents=True, exist_ok=True)
reference = Path('/content/drive/MyDrive/cedhai/cloud/cloud-evaluation-v1/latest.pt')
if not reference.is_file():
    raise FileNotFoundError('Reviewed checkpoint not found in Drive; do not rerun the old games')
print('Persistent experiment:', run_root)


In [ ]:
import hashlib, json, shutil, zipfile
from pathlib import PurePosixPath
uploaded_bundle = Path('/content/cedhai-next.zip')
persistent_bundle = run_root / 'source-bundle.zip'
if uploaded_bundle.exists():
    if persistent_bundle.exists() and hashlib.sha256(uploaded_bundle.read_bytes()).digest() != hashlib.sha256(persistent_bundle.read_bytes()).digest():
        raise ValueError('Different bundle: use the original bundle to resume or choose a new RUN_NAME')
    if not persistent_bundle.exists():
        shutil.copyfile(uploaded_bundle, persistent_bundle)
if not persistent_bundle.exists():
    raise FileNotFoundError('Upload runs/cedhai-next.zip using Explorer > Upload to Colab')
workspace = Path('/content/cedhai-next')
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(persistent_bundle) as archive:
    metadata = json.loads(archive.read('bundle.json'))
    names = archive.namelist()
    if len(names) != len(set(names)) or set(names) != set(metadata['sha256']) | {'bundle.json'} or metadata['schema_version'] != 1:
        raise ValueError('Invalid bundle')
    for name, expected in metadata['sha256'].items():
        parts = PurePosixPath(name).parts
        if not parts or parts[0] not in ('project', 'dataset') or '..' in parts or '\\' in name:
            raise ValueError('Unsafe bundle path')
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError('Bundle checksum mismatch')
        target = workspace.joinpath(*parts)
        if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError('Extracted code/data changed; restart the runtime with the original bundle')
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
print('Code revision:', metadata['git_revision'])


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(workspace / 'project') + '[training]'], check=True)
subprocess.run([sys.executable, '-m', 'cedhai.cloud_setup'], check=True)
def stage(name):
    command = [sys.executable, '-u', '-m', 'cedhai.next_experiment', '--root', str(run_root),
               '--reference', str(reference), '--stage', name]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise


## 1. Collect Independent Data
Seeds 21-26, four seating rotations each, using develop/random/random/defend scripts. Existing completed games are skipped on resume. Natural completion is required before training.


In [ ]:
stage('collect')


## 2. Audit And Train
Exports actor transitions, audits integrity and choice coverage, then trains from a fresh seeded initialization for 20 epochs. Forced/random labels are excluded. Validation seed 25 is small; audit warnings remain important. Model/optimizer checkpoints are saved each epoch. If interrupted, rerun this stage to resume.


In [ ]:
stage('train')
state = json.loads((run_root / 'state.json').read_text())
print('Candidate checkpoint:', state['candidate'])


## 3. Paired Gameplay Evaluation
Reference and candidate each play twenty games, using seeds 31,32,33,34,36 and identical seat rotations/opponents. Training/development seeds are excluded. This changes both dataset size and training budget; it is not a data-only causal experiment. Do not tune the candidate repeatedly on these evaluation results.


In [ ]:
stage('evaluate')


## 4. Export A Comparison
Download all three files from the Colab Contents root: reference-summary.zip, candidate-summary.zip, comparison.json. Persistent copies remain in the expanded-v1 Drive folder. No browser download widgets are needed.


In [ ]:
stage('export')
for name in ('reference-summary.zip', 'candidate-summary.zip', 'comparison.json'):
    path = Path('/content') / name
    print('Download:', path, '| bytes:', path.stat().st_size)
print((Path('/content/comparison.json')).read_text())
